# ML Coding Interview Prep: Complete Practice Notebook
## GM Staff AI/ML Engineer (Embodied AI) + Apple ML Eval

**What this covers:**
- Level 0: Broadcasting & Tensor Shape Fundamentals
- Level 1: Core Vectorized Patterns
- Block 1: Spatial & Geometric Logic (with solutions)
- Block 2: Architecture & Sequence Components (with solutions)
- Block 3: High-Efficiency ML Operations (with solutions)

**How to use:**
1. Read the concept section
2. Try the problem yourself FIRST (collapse the solution cell)
3. Compare with the reference solution
4. Run the test cases

**Time budget:** ~6-8 hours total


## Setup

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print("Setup complete!")


PyTorch version: 1.10.1
Setup complete!


---
# LEVEL 0: Broadcasting & Tensor Shape Fundamentals (~45 min)

## The Broadcasting Rules

PyTorch (and NumPy) compare shapes **element-wise from the trailing dimensions**:

1. If tensors have different number of dims, the shorter one is padded with 1s on the LEFT
2. Dimensions of size 1 are stretched to match the other tensor
3. Dimensions must either match or one of them must be 1 — otherwise ERROR

### Mental model:
Think of broadcasting as "virtual replication" — no memory is actually allocated.
`(3,1) + (1,4)` behaves like copying the column 4 times and the row 3 times → `(3,4)`


### Exercise 0.1: Shape Prediction Drills
For each pair, predict: does it broadcast? What's the output shape?

Write your answers before running the cell below.

```
(3, 4)       + (4,)           → ?
(3, 4)       + (3, 1)         → ?
(3, 4)       + (3,)           → ?
(2, 3, 4)    + (3, 4)         → ?
(2, 3, 4)    + (2, 1, 4)      → ?
(2, 3, 4)    + (1, 3, 1)      → ?
(5, 1, 3)    * (1, 4, 3)      → ?
(5, 1, 3)    * (1, 4, 1)      → ?
(8, 1, 6, 1) + (7, 1, 5)      → ?
(2, 1)       + (8, 4, 3)      → ?
```


In [2]:
# Exercise 0.1: Verify your predictions
pairs = [
    ((3, 4),       (4,)),
    ((3, 4),       (3, 1)),
    ((3, 4),       (3,)),       # This one ERRORS
    ((2, 3, 4),    (3, 4)),
    ((2, 3, 4),    (2, 1, 4)),
    ((2, 3, 4),    (1, 3, 1)),
    ((5, 1, 3),    (1, 4, 3)),
    ((5, 1, 3),    (1, 4, 1)),
    ((8, 1, 6, 1), (7, 1, 5)),
    ((2, 1),       (8, 4, 3)),  # This one ERRORS
]

for sa, sb in pairs:
    try:
        a = torch.randn(sa)
        b = torch.randn(sb)
        result = a + b
        print(f"{str(sa):20s} + {str(sb):15s} → {tuple(result.shape)}")
    except RuntimeError as e:
        print(f"{str(sa):20s} + {str(sb):15s} → ERROR: {e}")


(3, 4)               + (4,)            → (3, 4)
(3, 4)               + (3, 1)          → (3, 4)
(3, 4)               + (3,)            → ERROR: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1
(2, 3, 4)            + (3, 4)          → (2, 3, 4)
(2, 3, 4)            + (2, 1, 4)       → (2, 3, 4)
(2, 3, 4)            + (1, 3, 1)       → (2, 3, 4)
(5, 1, 3)            + (1, 4, 3)       → (5, 4, 3)
(5, 1, 3)            + (1, 4, 1)       → (5, 4, 3)
(8, 1, 6, 1)         + (7, 1, 5)       → (8, 7, 6, 5)
(2, 1)               + (8, 4, 3)       → ERROR: The size of tensor a (2) must match the size of tensor b (4) at non-singleton dimension 1


### Key Takeaways:
- `(3, 4) + (3,)` fails because trailing dim is 4 vs 3 — neither is 1
- `(2, 1) + (8, 4, 3)` fails because after left-padding → `(1, 2, 1) + (8, 4, 3)` → dim 1: 2 vs 4 FAIL
- The `(8, 1, 6, 1) + (7, 1, 5)` case: left-pad → `(8, 1, 6, 1) + (1, 7, 1, 5)` → `(8, 7, 6, 5)`


## Exercise 0.2: The Unsqueeze Pattern for Pairwise Operations

**Core idea:** To compute something between every pair from set A (size M) and set B (size N),
reshape A to `(M, 1, ...)` and B to `(1, N, ...)` so broadcasting creates the `(M, N, ...)` result.

This is THE most important pattern for interview problems.


In [3]:
# Exercise 0.2a: Pairwise differences
# Given two 1D tensors, compute all pairwise differences

a = torch.tensor([1.0, 2.0, 3.0])     # (3,)
b = torch.tensor([10.0, 20.0])         # (2,)

# YOUR APPROACH: Think about what shapes you need before coding
# We want output shape (3, 2) where result[i,j] = a[i] - b[j]

# Step 1: Make a shape (3, 1)
a_expanded = a.unsqueeze(1)    # or a[:, None] or a.view(-1, 1)
print(f"a_expanded shape: {a_expanded.shape}")

# Step 2: Make b shape (1, 2) — already broadcasts from (2,)
# Step 3: Subtract
diffs = a_expanded - b  # (3, 1) - (2,) → (3, 1) - (1, 2) → (3, 2)
print(f"diffs shape: {diffs.shape}")
print(f"diffs:\n{diffs}")
print(f"\nVerify: a[0]-b[1] = {a[0]-b[1]}, diffs[0,1] = {diffs[0,1]}")


a_expanded shape: torch.Size([3, 1])
diffs shape: torch.Size([3, 2])
diffs:
tensor([[ -9., -19.],
        [ -8., -18.],
        [ -7., -17.]])

Verify: a[0]-b[1] = -19.0, diffs[0,1] = -19.0


In [71]:
# Exercise 0.2b: Pairwise Euclidean distances between 2D point sets
# points_a: (M, 2), points_b: (N, 2)
# Output: (M, N) distance matrix

points_a = torch.tensor([[0.0, 0.0], [1.0, 1.0], [2.0, 0.0]])  # (3, 2)
points_b = torch.tensor([[0.0, 1.0], [1.0, 0.0]])                # (2, 2)

# YOUR TURN: Try it yourself first before looking below

# Shape analysis:
# points_a: (3, 2) → unsqueeze(1) → (3, 1, 2)
# points_b: (2, 2) → unsqueeze(0) → (1, 2, 2)
# diff: (3, 1, 2) - (1, 2, 2) → (3, 2, 2)
# squared sum over last dim → (3, 2)
# sqrt → (3, 2) ✓

diff = points_a.unsqueeze(1) - points_b.unsqueeze(0)  # (3, 1, 2) - (1, 2, 2) → (3, 2, 2)
print(f"diff shape: {diff.shape}")

dist = torch.sqrt((diff ** 2).sum(dim=-1))  # (3, 2, 2) → sum over D → (3, 2)
print(f"dist shape: {dist.shape}")
print(f"dist:\n{dist}")

# Verify: dist[0,0] = distance from (0,0) to (0,1) = 1.0
print(f"\nVerify dist[0,0] = {dist[0,0]:.4f} (expected 1.0)")
print(f"Verify dist[1,1] = {dist[1,1]:.4f} (expected {math.sqrt(2):.4f})")


diff shape: torch.Size([3, 2, 2])
dist shape: torch.Size([3, 2])
dist:
tensor([[1.0000, 1.0000],
        [1.0000, 1.0000],
        [2.2361, 1.0000]])

Verify dist[0,0] = 1.0000 (expected 1.0)
Verify dist[1,1] = 1.0000 (expected 1.4142)


In [73]:
diff.shape

torch.Size([3, 2, 2])

In [ ]:
points_a.unsqueeze(1)[0]

tensor([[0., 0.]])

In [10]:
# Exercise 0.2c: Outer product via broadcasting
# Given vectors a (M,) and b (N,), compute outer product (M, N)
# WITHOUT using torch.outer or torch.einsum

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0])

outer = a.unsqueeze(1) * b.unsqueeze(0)  # (3,1) * (1,2) → (3,2)
print(f"Broadcasting outer product:\n{outer}")
print(f"torch.outer verification:\n{torch.outer(a, b)}")
assert torch.allclose(outer, torch.outer(a, b))
print("✓ Match!")


Broadcasting outer product:
tensor([[ 4.,  5.],
        [ 8., 10.],
        [12., 15.]])
torch.outer verification:
tensor([[ 4.,  5.],
        [ 8., 10.],
        [12., 15.]])
✓ Match!


## Exercise 0.3: Einsum — Your Swiss Army Knife

`torch.einsum` is incredibly useful in interviews for expressing complex tensor operations concisely.

**Syntax:** Each letter is a dimension index. Repeated letters mean those dims are multiplied/summed.


In [70]:
# Einsum examples — understand each one

A = torch.randn(3, 4)
B = torch.randn(4, 5)
v = torch.randn(4)

# Matrix multiply: sum over shared dim k
C = torch.einsum('ik,kj->ij', A, B)
print(f"matmul: {A.shape} x {B.shape} → {C.shape}")
assert torch.allclose(C, A @ B)

# Batch matrix multiply
A_batch = torch.randn(2, 3, 4)
B_batch = torch.randn(2, 4, 5)
C_batch = torch.einsum('bik,bkj->bij', A_batch, B_batch)
print(f"batch matmul: {A_batch.shape} x {B_batch.shape} → {C_batch.shape}")
assert torch.allclose(C_batch, torch.bmm(A_batch, B_batch))

# Dot product
dot = torch.einsum('i,i->', v, v)
print(f"dot product: {dot.item():.4f} vs {v.dot(v).item():.4f}")

# Outer product
outer = torch.einsum('i,j->ij', v[:3], v[:2])
print(f"outer product shape: {outer.shape}")

# Trace
sq = torch.randn(4, 4)
tr = torch.einsum('ii->', sq)
print(f"trace: {tr.item():.4f} vs {sq.trace().item():.4f}")

# Transpose
At = torch.einsum('ij->ji', A)
assert torch.allclose(At, A.T)

print("\n✓ All einsum examples verified!")


matmul: torch.Size([3, 4]) x torch.Size([4, 5]) → torch.Size([3, 5])
batch matmul: torch.Size([2, 3, 4]) x torch.Size([2, 4, 5]) → torch.Size([2, 3, 5])
dot product: 0.8604 vs 0.8604
outer product shape: torch.Size([3, 2])
trace: 1.4760 vs 1.4760

✓ All einsum examples verified!


In [67]:
# Exercise 0.3b: Attention score computation with einsum
# Q: (batch, heads, seq_q, d_k)
# K: (batch, heads, seq_k, d_k)
# We want: scores (batch, heads, seq_q, seq_k) = Q @ K^T per batch per head

batch, heads, seq_q, seq_k, d_k = 2, 4, 8, 10, 16

Q = torch.randn(batch, heads, seq_q, d_k)
K = torch.randn(batch, heads, seq_k, d_k)

# Method 1: einsum (clean and readable)
scores_einsum = torch.einsum('bhqd,bhkd->bhqk', Q, K)

# Method 2: transpose + matmul
scores_matmul = Q @ K.transpose(-2, -1)

print(f"Q: {Q.shape}, K: {K.shape}")
print(f"Scores (einsum): {scores_einsum.shape}")
print(f"Scores (matmul): {scores_matmul.shape}")
assert torch.allclose(scores_einsum, scores_matmul, atol=1e-5)
print("✓ Both methods match!")


Q: torch.Size([2, 4, 8, 16]), K: torch.Size([2, 4, 10, 16])
Scores (einsum): torch.Size([2, 4, 8, 10])
Scores (matmul): torch.Size([2, 4, 8, 10])
✓ Both methods match!


In [69]:
scores_einsum.shape

torch.Size([2, 4, 8, 10])

---
# LEVEL 1: Core Vectorized Patterns (~45 min)

These are the building-block patterns that appear inside almost every interview problem.


## Pattern 1.1: Pairwise Comparison Matrix

**When you see:** "for each element in A, compare with every element in B"
**Think:** unsqueeze + broadcast


In [49]:
# Pattern 1.1: Which boxes overlap which boxes?
# Given M boxes and N boxes, produce an (M, N) boolean overlap matrix

def boxes_overlap(boxes_a, boxes_b):
    """
    Args:
        boxes_a: (M, 4) [x1, y1, x2, y2]
        boxes_b: (N, 4) [x1, y1, x2, y2]
    Returns:
        (M, N) boolean tensor — True if boxes overlap
    """
    # Shape trick: (M, 1, 4) vs (1, N, 4)
    a = boxes_a.unsqueeze(1)  # (M, 1, 4)
    b = boxes_b.unsqueeze(0)  # (1, N, 4)
    
    # Two boxes overlap if they overlap in BOTH x and y
    # x overlap: a.x1 < b.x2 AND a.x2 > b.x1
    # y overlap: a.y1 < b.y2 AND a.y2 > b.y1
    
    x_overlap = (a[..., 0] < b[..., 2]) & (a[..., 2] > b[..., 0])  # (M, N)
    y_overlap = (a[..., 1] < b[..., 3]) & (a[..., 3] > b[..., 1])  # (M, N)
    
    return x_overlap & y_overlap  # (M, N)

# Test
a = torch.tensor([[0., 0., 2., 2.], [5., 5., 7., 7.]])  # 2 boxes
b = torch.tensor([[1., 1., 3., 3.], [10., 10., 12., 12.], [0., 0., 1., 1.]])  # 3 boxes

overlap = boxes_overlap(a, b)
print(f"Shape: {overlap.shape}")  # (2, 3)
print(f"Overlap matrix:\n{overlap}")
# Box 0 overlaps with b[0] and b[2], not b[1]
# Box 1 overlaps with none
print("✓ Pairwise overlap computed without any loops!")


Shape: torch.Size([2, 3])
Overlap matrix:
tensor([[ True, False,  True],
        [False, False, False]])
✓ Pairwise overlap computed without any loops!


In [56]:
a = a.unsqueeze(1)

In [57]:
b= b.unsqueeze(0)

In [58]:
a

tensor([[[0., 0., 2., 2.]],

        [[5., 5., 7., 7.]]])

In [59]:
b

tensor([[[ 1.,  1.,  3.,  3.],
         [10., 10., 12., 12.],
         [ 0.,  0.,  1.,  1.]]])

In [60]:
a.shape

torch.Size([2, 1, 4])

In [66]:
(a[...,0] < b[...,2]) & ((a[...,2] > b[...,0]))

tensor([[ True, False,  True],
        [False, False, False]])

## Pattern 1.2: Masked Fill / Conditional Assignment

**When you see:** "set values to X where condition Y holds"
**Think:** `torch.where()` or `tensor.masked_fill_()`


In [14]:
# Pattern 1.2: Masked operations
scores = torch.randn(3, 5)  # attention scores (seq_q, seq_k)

# Create causal mask: can't attend to future positions
# mask[i,j] = True means BLOCK position j when querying from i
causal_mask = torch.triu(torch.ones(3, 5, dtype=torch.bool), diagonal=1)
print(f"Causal mask:\n{causal_mask}")

# Method 1: masked_fill_ (in-place)
scores_masked = scores.clone()
scores_masked.masked_fill_(causal_mask, float('-inf'))
print(f"\nScores after masking:\n{scores_masked}")

# Method 2: torch.where (functional)
scores_masked2 = torch.where(causal_mask, torch.tensor(float('-inf')), scores)
assert torch.allclose(scores_masked, scores_masked2, equal_nan=True)

# After softmax, masked positions become 0
probs = F.softmax(scores_masked, dim=-1)
print(f"\nAttention probs (rows sum to 1):\n{probs}")
print(f"Row sums: {probs.sum(dim=-1)}")
print("✓ Masked positions contribute zero attention weight!")


Causal mask:
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True]])

Scores after masking:
tensor([[ 1.0860,    -inf,    -inf,    -inf,    -inf],
        [-0.7408,  0.8166,    -inf,    -inf,    -inf],
        [ 0.9253,  0.1879, -0.3279,    -inf,    -inf]])

Attention probs (rows sum to 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1740, 0.8260, 0.0000, 0.0000, 0.0000],
        [0.5669, 0.2712, 0.1619, 0.0000, 0.0000]])
Row sums: tensor([1.0000, 1.0000, 1.0000])
✓ Masked positions contribute zero attention weight!


In [48]:

torch.triu(torch.ones(3, 5, dtype=torch.bool), diagonal=0)

tensor([[ True,  True,  True,  True,  True],
        [False,  True,  True,  True,  True],
        [False, False,  True,  True,  True]])

In [41]:
scores = torch.randn(3, 5)  # attention scores (seq_q, seq_k)


In [42]:
scores

tensor([[ 0.5349,  0.8094,  1.1103, -1.6898, -0.9890],
        [ 0.9580,  1.3221,  0.8172, -0.7658, -0.7506],
        [ 1.3525,  0.6863, -0.3278,  0.7950,  0.2815]])

In [43]:
causal_mask = torch.triu(torch.ones(3, 5, dtype=torch.bool), diagonal=1)

In [44]:
causal_mask

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True]])

In [45]:
# Method 1: masked_fill_ (in-place)
scores_masked = scores.clone()
scores_masked.masked_fill_(causal_mask, float('-inf'))
print(f"\nScores after masking:\n{scores_masked}")


Scores after masking:
tensor([[ 0.5349,    -inf,    -inf,    -inf,    -inf],
        [ 0.9580,  1.3221,    -inf,    -inf,    -inf],
        [ 1.3525,  0.6863, -0.3278,    -inf,    -inf]])


In [46]:
scores_masked

tensor([[ 0.5349,    -inf,    -inf,    -inf,    -inf],
        [ 0.9580,  1.3221,    -inf,    -inf,    -inf],
        [ 1.3525,  0.6863, -0.3278,    -inf,    -inf]])

In [47]:
probs = F.softmax(scores_masked, dim=-1)
print(f"\nAttention probs (rows sum to 1):\n{probs}")
print(f"Row sums: {probs.sum(dim=-1)}")
print("✓ Masked positions contribute zero attention weight!")


Attention probs (rows sum to 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4100, 0.5900, 0.0000, 0.0000, 0.0000],
        [0.5882, 0.3022, 0.1096, 0.0000, 0.0000]])
Row sums: tensor([1., 1., 1.])
✓ Masked positions contribute zero attention weight!


## Pattern 1.3: Scatter / Gather for Grouping

**When you see:** "aggregate by group" or "gather from indices"
**Think:** `scatter_add_`, `gather`, `index_select`


In [ ]:
# Pattern 1.3a: Aggregate embeddings by group ID (no loops!)
embeddings = torch.tensor([
    [1.0, 0.0],   # token 0, group 0
    [2.0, 0.0],   # token 1, group 0  
    [0.0, 3.0],   # token 2, group 1
    [0.0, 1.0],   # token 3, group 1
    [5.0, 5.0],   # token 4, group 2
])
group_ids = torch.tensor([0, 0, 1, 1, 2])
num_groups = 3

# Sum per group using scatter_add_
# Need to expand group_ids to match embedding dims
ids_expanded = group_ids.unsqueeze(1).expand(-1, embeddings.shape[1])  # (5, 2)
print(f"ids_expanded shape: {ids_expanded.shape}")

group_sums = torch.zeros(num_groups, embeddings.shape[1])
group_sums.scatter_add_(0, ids_expanded, embeddings)
print(f"Group sums:\n{group_sums}")

# Count per group for mean
counts = torch.bincount(group_ids, minlength=num_groups).float().unsqueeze(1)  # (3, 1)
group_means = group_sums / counts
print(f"Group means:\n{group_means}")
# Expected: group 0 = [1.5, 0], group 1 = [0, 2], group 2 = [5, 5]


ids_expanded shape: torch.Size([5, 2])
Group sums:
tensor([[3., 0.],
        [0., 4.],
        [5., 5.]])
Group means:
tensor([[1.5000, 0.0000],
        [0.0000, 2.0000],
        [5.0000, 5.0000]])


In [21]:
embeddings = torch.tensor([
    [1.0, 0.0],   # token 0, group 0
    [2.0, 0.0],   # token 1, group 0  
    [0.0, 3.0],   # token 2, group 1
    [0.0, 1.0],   # token 3, group 1
    [5.0, 5.0],   # token 4, group 2
])

In [22]:
embeddings.shape

torch.Size([5, 2])

In [23]:
group_ids = torch.tensor([0, 0, 1, 1, 2])
num_groups = 3


In [24]:
group_ids.shape

torch.Size([5])

In [36]:
ids_expanded = group_ids.unsqueeze(1).expand(-1, embeddings.shape[1])  # (5, 2)


In [39]:
group_sums = torch.zeros(num_groups, embeddings.shape[1])


In [40]:
group_sums

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

In [33]:
group_sums.shape

torch.Size([3, 2])

In [34]:
num_groups

3

In [37]:
group_sums.scatter_add_(0, ids_expanded, embeddings)

tensor([[3., 0.],
        [0., 4.],
        [5., 5.]])

In [38]:
group_sums

tensor([[3., 0.],
        [0., 4.],
        [5., 5.]])

In [4]:
# Pattern 1.3b: Gather — select specific elements per row
# Common in: beam search, top-k selection, indexing attention outputs

# Scenario: for each query, we know which key index to attend to
values = torch.tensor([
    [10., 20., 30., 40.],
    [50., 60., 70., 80.],
    [90., 100., 110., 120.],
])  # (3, 4) — 3 queries, 4 possible values each

indices = torch.tensor([
    [2],   # query 0 selects value at index 2 → 30
    [0],   # query 1 selects value at index 0 → 50
    [3],   # query 2 selects value at index 3 → 120
])  # (3, 1)

gathered = torch.gather(values, dim=1, index=indices)
print(f"Gathered: {gathered.squeeze().tolist()}")  # [30, 50, 120]

# Top-k version: select top-2 per row
topk_vals, topk_idx = torch.topk(values, k=2, dim=1)
print(f"\nTop-2 values:\n{topk_vals}")
print(f"Top-2 indices:\n{topk_idx}")


Gathered: [30.0, 50.0, 120.0]

Top-2 values:
tensor([[ 40.,  30.],
        [ 80.,  70.],
        [120., 110.]])
Top-2 indices:
tensor([[3, 2],
        [3, 2],
        [3, 2]])


In [5]:
values.shape, indices.shape

(torch.Size([3, 4]), torch.Size([3, 1]))

In [6]:
gathered = torch.gather(values, dim=1, index=indices)

In [7]:
gathered.shape

torch.Size([3, 1])

In [8]:
gathered

tensor([[ 30.],
        [ 50.],
        [120.]])

In [3]:
# Pattern 1.3c: Beam Search — maintain top-k sequences across decoding steps
# Common in: text generation, translation, seq2seq decoding
# Key ops: topk to select candidates, gather to reorder beam history

# Toy setup: vocab of 5 tokens, beam width of 2, max 3 steps
# Instead of a real model, we use fixed log-prob tables per step
vocab_size = 5
beam_width = 2
vocab = ["<s>", "cat", "sat", "on", "mat"]

# Simulated log-probs at each decoding step: shape (beam_width, vocab_size)
# Step 0: only 1 active beam (start token), so shape is (1, vocab_size)
step0_logprobs = torch.tensor([[-1.6, -0.5, -2.3, -1.9, -0.9]])  # (1, vocab)
step1_logprobs = torch.tensor([[-2.1, -0.4, -1.8, -2.5, -0.6],   # beam 0 continuations
                               [-1.5, -2.2, -0.3, -1.1, -2.8]])  # beam 1 continuations
step2_logprobs = torch.tensor([[-1.9, -2.4, -0.2, -1.3, -2.1],
                               [-0.8, -1.7, -2.5, -0.4, -1.2]])

# ── Step 0: initialize beams from the single start beam ──────────────────
beam_scores, beam_tokens = torch.topk(step0_logprobs[0], k=beam_width)
beam_sequences = beam_tokens.unsqueeze(1)  # (beam_width, seq_len=1)

print("After step 0:")
for b in range(beam_width):
    print(f"  Beam {b}: [{vocab[beam_tokens[b]]}]  score={beam_scores[b]:.2f}")

# ── Step 1: expand each beam, score all candidates, keep global top-k ────
# candidate_scores[i, j] = beam_scores[i] + step1_logprobs[i, j]
candidate_scores = beam_scores.unsqueeze(1) + step1_logprobs  # (beam_width, vocab)

# Flatten to find global top-k across all (beam, token) pairs
flat_scores = candidate_scores.view(-1)                        # (beam_width * vocab,)
top_scores, top_flat_idx = torch.topk(flat_scores, k=beam_width)

# Recover which beam and which token each winner came from
parent_beams = top_flat_idx // vocab_size   # which beam it extended
next_tokens  = top_flat_idx  % vocab_size   # which token was chosen

# Use gather to pull the correct beam history forward (the gather connection!)
beam_sequences = torch.gather(
    beam_sequences,
    dim=0,
    index=parent_beams.unsqueeze(1).expand(-1, beam_sequences.shape[1])
)  # reorder rows so each surviving beam has its correct history
beam_sequences = torch.cat([beam_sequences, next_tokens.unsqueeze(1)], dim=1)
beam_scores = top_scores

print("\nAfter step 1:")
for b in range(beam_width):
    tokens = [vocab[t] for t in beam_sequences[b].tolist()]
    print(f"  Beam {b}: {tokens}  score={beam_scores[b]:.2f}  (extended beam {parent_beams[b].item()})")

# ── Step 2: same expansion logic ─────────────────────────────────────────
candidate_scores = beam_scores.unsqueeze(1) + step2_logprobs
flat_scores = candidate_scores.view(-1)
top_scores, top_flat_idx = torch.topk(flat_scores, k=beam_width)
parent_beams = top_flat_idx // vocab_size
next_tokens  = top_flat_idx  % vocab_size

beam_sequences = torch.gather(
    beam_sequences, dim=0,
    index=parent_beams.unsqueeze(1).expand(-1, beam_sequences.shape[1])
)
beam_sequences = torch.cat([beam_sequences, next_tokens.unsqueeze(1)], dim=1)
beam_scores = top_scores

print("\nFinal beams (best first):")
for b in range(beam_width):
    tokens = [vocab[t] for t in beam_sequences[b].tolist()]
    print(f"  Beam {b}: {tokens}  score={beam_scores[b]:.2f}")

print(f"\nBest sequence: {[vocab[t] for t in beam_sequences[0].tolist()]}")

# Shape checkpoint:
# beam_sequences:   (beam_width, seq_len)   — grows by 1 each step
# candidate_scores: (beam_width, vocab_size) — flattened for global top-k
# parent_beams:     (beam_width,)            — which row to gather from history


After step 0:
  Beam 0: [cat]  score=-0.50
  Beam 1: [mat]  score=-0.90

After step 1:
  Beam 0: ['cat', 'cat']  score=-0.90  (extended beam 0)
  Beam 1: ['cat', 'mat']  score=-1.10  (extended beam 0)

Final beams (best first):
  Beam 0: ['cat', 'cat', 'sat']  score=-1.10
  Beam 1: ['cat', 'mat', 'on']  score=-1.50

Best sequence: ['cat', 'cat', 'sat']


/var/folders/yz/yvgt5l4x3v7fhczkxn337rph0000gn/T/ipykernel_38822/3870559499.py:36: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  parent_beams = top_flat_idx // vocab_size   # which beam it extended
/var/folders/yz/yvgt5l4x3v7fhczkxn337rph0000gn/T/ipykernel_38822/3870559499.py:57: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  parent_be

## Pattern 1.4: Reduction with Keepdim

**Critical for shape management.** `keepdim=True` preserves the reduced dimension as size 1,
enabling further broadcasting.


In [17]:
# Pattern 1.4: keepdim usage
x = torch.randn(2, 3, 4)

# Without keepdim
mean_no_keep = x.mean(dim=-1)
print(f"Without keepdim: {x.shape} → mean → {mean_no_keep.shape}")  # (2, 3)

# With keepdim  
mean_keep = x.mean(dim=-1, keepdim=True)
print(f"With keepdim:    {x.shape} → mean → {mean_keep.shape}")  # (2, 3, 1)

# Why it matters: centering data
x_centered = x - mean_keep  # (2,3,4) - (2,3,1) → broadcasts correctly!
print(f"Centered shape: {x_centered.shape}")

# Without keepdim this would fail:
try:
    x_bad = x - mean_no_keep  # (2,3,4) - (2,3) → ERROR
except RuntimeError as e:
    print(f"\nWithout keepdim: ERROR — {e}")

# Softmax manually with keepdim for numerical stability
logits = torch.randn(2, 5)
max_vals = logits.max(dim=-1, keepdim=True).values  # (2, 1)
stable_exp = torch.exp(logits - max_vals)            # (2, 5) - (2, 1) → (2, 5)
softmax = stable_exp / stable_exp.sum(dim=-1, keepdim=True)  # (2, 5) / (2, 1) → (2, 5)
print(f"\nManual softmax matches F.softmax: {torch.allclose(softmax, F.softmax(logits, dim=-1))}")


Without keepdim: torch.Size([2, 3, 4]) → mean → torch.Size([2, 3])
With keepdim:    torch.Size([2, 3, 4]) → mean → torch.Size([2, 3, 1])
Centered shape: torch.Size([2, 3, 4])

Without keepdim: ERROR — The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 2

Manual softmax matches F.softmax: True


In [10]:
logits = torch.randn(2, 5)

In [12]:
logits.shape

torch.Size([2, 5])

In [13]:
max_vals = logits.max(dim=-1, keepdim=True).values  # (2, 1)

In [14]:
max_vals

tensor([[0.3367],
        [2.2082]])

In [15]:
logits

tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229],
        [-0.1863,  2.2082, -0.6380,  0.4617,  0.2674]])

In [16]:
stable_exp = torch.exp(logits - max_vals)

In [17]:
stable_exp

tensor([[1.0000, 0.8123, 0.9028, 0.8991, 0.2323],
        [0.0912, 1.0000, 0.0581, 0.1744, 0.1436]])

In [19]:
softmax = stable_exp / stable_exp.sum(dim=-1, keepdim=True)

In [20]:
softmax

tensor([[0.2600, 0.2112, 0.2347, 0.2337, 0.0604],
        [0.0622, 0.6816, 0.0396, 0.1188, 0.0979]])

In [21]:
x = torch.randn(2, 3, 4)

In [23]:
x.shape

torch.Size([2, 3, 4])

In [30]:
x

tensor([[[ 0.3449,  0.7991,  0.3431, -0.7109],
         [ 0.4065,  0.9562,  0.3075,  0.3181],
         [ 2.5384,  0.8318,  0.5719,  0.0898]],

        [[-2.1488, -0.2669,  0.0645, -1.4933],
         [-1.7842, -1.5064, -0.5167,  2.0545],
         [ 1.2938,  0.8170, -1.0389, -1.4859]]])

In [38]:
x - x.mean(dim=-1,keepdim=True)

tensor([[[ 0.1508,  0.6051,  0.1491, -0.9049],
         [-0.0905,  0.4591, -0.1896, -0.1790],
         [ 1.5305, -0.1762, -0.4361, -0.9182]],

        [[-1.1877,  0.6942,  1.0256, -0.5322],
         [-1.3460, -1.0682, -0.0784,  2.4927],
         [ 1.3973,  0.9205, -0.9354, -1.3824]]])

---
# BLOCK 1: Spatial & Geometric Logic (~2 hours)

These are the highest-priority problems for the Embodied AI role.


## Problem 1A: Vectorized 2D IoU Matrix (30 min)

**Prompt:** Given two sets of axis-aligned bounding boxes, compute the pairwise IoU matrix.

- `boxes_a`: `(M, 4)` → `[x1, y1, x2, y2]`
- `boxes_b`: `(N, 4)` → `[x1, y1, x2, y2]`
- Return: `(M, N)` IoU matrix

**No loops. Fully vectorized.**


In [ ]:
# ============ YOUR SOLUTION HERE ============
def iou_matrix(boxes_a: torch.Tensor, boxes_b: torch.Tensor) -> torch.Tensor:
    """
    Compute pairwise IoU between two sets of axis-aligned boxes.
    
    Args:
        boxes_a: (M, 4) tensor [x1, y1, x2, y2]
        boxes_b: (N, 4) tensor [x1, y1, x2, y2]
    Returns:
        (M, N) tensor of IoU values
    """
    # TRY IT YOURSELF FIRST — then check the solution below
    pass


In [74]:
# ============ REFERENCE SOLUTION ============
def iou_matrix(boxes_a: torch.Tensor, boxes_b: torch.Tensor) -> torch.Tensor:
    """
    Compute pairwise IoU between two sets of axis-aligned boxes.
    
    Args:
        boxes_a: (M, 4) tensor [x1, y1, x2, y2]
        boxes_b: (N, 4) tensor [x1, y1, x2, y2]
    Returns:
        (M, N) tensor of IoU values
        
    Complexity: O(M*N) time, O(M*N) space
    """
    # Step 1: Expand for pairwise computation
    # boxes_a: (M, 4) → (M, 1, 4)
    # boxes_b: (N, 4) → (1, N, 4)
    a = boxes_a.unsqueeze(1)  # (M, 1, 4)
    b = boxes_b.unsqueeze(0)  # (1, N, 4)
    
    # Step 2: Intersection coordinates
    # For each (i, j) pair, find the overlap rectangle
    inter_x1 = torch.max(a[..., 0], b[..., 0])  # (M, N)
    inter_y1 = torch.max(a[..., 1], b[..., 1])  # (M, N)
    inter_x2 = torch.min(a[..., 2], b[..., 2])  # (M, N)
    inter_y2 = torch.min(a[..., 3], b[..., 3])  # (M, N)
    
    # Step 3: Intersection area (clamp to 0 if no overlap)
    inter_width = (inter_x2 - inter_x1).clamp(min=0)   # (M, N)
    inter_height = (inter_y2 - inter_y1).clamp(min=0)   # (M, N)
    inter_area = inter_width * inter_height              # (M, N)
    
    # Step 4: Individual box areas
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])  # (M,)
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])  # (N,)
    
    # Step 5: Union = area_a + area_b - intersection
    # area_a: (M,) → (M, 1) for broadcasting with (M, N)
    union = area_a.unsqueeze(1) + area_b.unsqueeze(0) - inter_area  # (M, N)
    
    # Step 6: IoU (avoid division by zero)
    iou = inter_area / (union + 1e-6)  # (M, N)
    
    return iou

# ============ TEST CASES ============
print("=== Test 1: Identical boxes ===")
a = torch.tensor([[0., 0., 2., 2.]])
b = torch.tensor([[0., 0., 2., 2.]])
result = iou_matrix(a, b)
print(f"IoU: {result.item():.4f} (expected 1.0)")
assert torch.isclose(result, torch.tensor([[1.0]]), atol=1e-4)

print("\n=== Test 2: No overlap ===")
a = torch.tensor([[0., 0., 1., 1.]])
b = torch.tensor([[2., 2., 3., 3.]])
result = iou_matrix(a, b)
print(f"IoU: {result.item():.4f} (expected 0.0)")
assert result.item() < 1e-5

print("\n=== Test 3: Partial overlap ===")
a = torch.tensor([[0., 0., 2., 2.]])
b = torch.tensor([[1., 1., 3., 3.]])
result = iou_matrix(a, b)
expected = 1.0 / 7.0
print(f"IoU: {result.item():.4f} (expected {expected:.4f})")
assert torch.isclose(result, torch.tensor([[expected]]), atol=1e-3)

print("\n=== Test 4: Batch shapes ===")
a = torch.tensor([[0.,0.,2.,2.], [1.,1.,3.,3.]])
b = torch.tensor([[0.,0.,1.,1.], [1.,1.,2.,2.], [0.,0.,3.,3.]])
result = iou_matrix(a, b)
print(f"Shape: {result.shape} (expected (2, 3))")
print(f"IoU matrix:\n{result}")
assert result.shape == (2, 3)

print("\n✓ All IoU tests passed!")


=== Test 1: Identical boxes ===
IoU: 1.0000 (expected 1.0)

=== Test 2: No overlap ===
IoU: 0.0000 (expected 0.0)

=== Test 3: Partial overlap ===
IoU: 0.1429 (expected 0.1429)

=== Test 4: Batch shapes ===
Shape: torch.Size([2, 3]) (expected (2, 3))
IoU matrix:
tensor([[0.2500, 0.2500, 0.4444],
        [0.0000, 0.2500, 0.4444]])

✓ All IoU tests passed!


## Problem 1B: Non-Maximum Suppression (30 min)

**Prompt:** Implement greedy NMS. Use your `iou_matrix` from 1A.

The greedy loop is inherently sequential (each step depends on the previous),
but IoU computation within each step should be vectorized.


In [ ]:
# ============ YOUR SOLUTION HERE ============
def nms(boxes: torch.Tensor, scores: torch.Tensor, iou_threshold: float) -> torch.Tensor:
    """
    Greedy Non-Maximum Suppression.
    
    Args:
        boxes: (N, 4) tensor [x1, y1, x2, y2]
        scores: (N,) confidence scores
        iou_threshold: suppress boxes with IoU > this
    Returns:
        keep: 1-D tensor of kept indices, sorted by descending score
    """
    # TRY IT YOURSELF
    pass


In [75]:
# ============ REFERENCE SOLUTION ============
def nms(boxes: torch.Tensor, scores: torch.Tensor, iou_threshold: float) -> torch.Tensor:
    """
    Greedy Non-Maximum Suppression.
    
    Args:
        boxes: (N, 4) tensor [x1, y1, x2, y2]
        scores: (N,) confidence scores
        iou_threshold: suppress boxes with IoU > this
    Returns:
        keep: 1-D tensor of kept indices, sorted by descending score
        
    Complexity: O(N^2) worst case for IoU computation, O(N) iterations
    """
    # Sort by descending score
    order = scores.argsort(descending=True)  # (N,)
    
    keep = []
    
    while order.numel() > 0:
        # Pick the highest-scoring remaining box
        current = order[0].item()
        keep.append(current)
        
        if order.numel() == 1:
            break
        
        # Compute IoU of current box with all remaining
        remaining = order[1:]
        ious = iou_matrix(
            boxes[current].unsqueeze(0),   # (1, 4)
            boxes[remaining]                # (R, 4)
        ).squeeze(0)                        # (R,)
        
        # Keep boxes with IoU <= threshold (not suppressed)
        mask = ious <= iou_threshold  # (R,) boolean
        order = remaining[mask]
    
    return torch.tensor(keep, dtype=torch.long)

# ============ TEST CASES ============
print("=== NMS Test 1: Basic suppression ===")
boxes = torch.tensor([
    [0., 0., 2., 2.],
    [0.1, 0.1, 2.1, 2.1],  # overlaps heavily with box 0
    [5., 5., 7., 7.],       # no overlap
])
scores = torch.tensor([0.9, 0.8, 0.7])
keep = nms(boxes, scores, iou_threshold=0.5)
print(f"Kept indices: {keep.tolist()} (expected [0, 2])")
assert keep.tolist() == [0, 2]

print("\n=== NMS Test 2: No suppression ===")
boxes = torch.tensor([
    [0., 0., 1., 1.],
    [3., 3., 4., 4.],
    [6., 6., 7., 7.],
])
scores = torch.tensor([0.5, 0.9, 0.3])
keep = nms(boxes, scores, iou_threshold=0.5)
print(f"Kept indices: {keep.tolist()} (expected [1, 0, 2])")
assert len(keep) == 3

print("\n=== NMS Test 3: All suppressed except one ===")
boxes = torch.tensor([
    [0., 0., 2., 2.],
    [0.05, 0.05, 2.05, 2.05],
    [0.1, 0.1, 2.1, 2.1],
])
scores = torch.tensor([0.9, 0.8, 0.7])
keep = nms(boxes, scores, iou_threshold=0.5)
print(f"Kept indices: {keep.tolist()} (expected [0])")
assert keep.tolist() == [0]

print("\n✓ All NMS tests passed!")


=== NMS Test 1: Basic suppression ===
Kept indices: [0, 2] (expected [0, 2])

=== NMS Test 2: No suppression ===
Kept indices: [1, 0, 2] (expected [1, 0, 2])

=== NMS Test 3: All suppressed except one ===
Kept indices: [0] (expected [0])

✓ All NMS tests passed!


## Problem 1C: 3D IoU for Embodied AI (25 min)

Extend IoU to 3D axis-aligned bounding boxes. This is critical for LiDAR-based perception.
If you got 1A right, this is straightforward — just add the z-dimension.


In [76]:
# ============ REFERENCE SOLUTION ============
def iou_3d(boxes_a: torch.Tensor, boxes_b: torch.Tensor) -> torch.Tensor:
    """
    Compute pairwise 3D IoU between axis-aligned 3D boxes.
    
    Args:
        boxes_a: (M, 6) tensor [x1, y1, z1, x2, y2, z2]
        boxes_b: (N, 6) tensor [x1, y1, z1, x2, y2, z2]
    Returns:
        (M, N) IoU matrix
    """
    a = boxes_a.unsqueeze(1)  # (M, 1, 6)
    b = boxes_b.unsqueeze(0)  # (1, N, 6)
    
    # Intersection in each dimension
    inter_x1 = torch.max(a[..., 0], b[..., 0])  # (M, N)
    inter_y1 = torch.max(a[..., 1], b[..., 1])
    inter_z1 = torch.max(a[..., 2], b[..., 2])
    inter_x2 = torch.min(a[..., 3], b[..., 3])
    inter_y2 = torch.min(a[..., 4], b[..., 4])
    inter_z2 = torch.min(a[..., 5], b[..., 5])
    
    # Intersection volume
    inter_vol = (
        (inter_x2 - inter_x1).clamp(min=0) *
        (inter_y2 - inter_y1).clamp(min=0) *
        (inter_z2 - inter_z1).clamp(min=0)
    )  # (M, N)
    
    # Individual volumes
    vol_a = (
        (boxes_a[:, 3] - boxes_a[:, 0]) *
        (boxes_a[:, 4] - boxes_a[:, 1]) *
        (boxes_a[:, 5] - boxes_a[:, 2])
    )  # (M,)
    vol_b = (
        (boxes_b[:, 3] - boxes_b[:, 0]) *
        (boxes_b[:, 4] - boxes_b[:, 1]) *
        (boxes_b[:, 5] - boxes_b[:, 2])
    )  # (N,)
    
    union = vol_a.unsqueeze(1) + vol_b.unsqueeze(0) - inter_vol  # (M, N)
    
    return inter_vol / (union + 1e-6)

# Test
a = torch.tensor([[0., 0., 0., 2., 2., 2.]])  # 2x2x2 cube at origin
b = torch.tensor([[1., 1., 1., 3., 3., 3.]])  # 2x2x2 cube shifted by (1,1,1)
result = iou_3d(a, b)
# Intersection: 1x1x1 = 1, Union: 8+8-1 = 15, IoU = 1/15
print(f"3D IoU: {result.item():.4f} (expected {1/15:.4f})")
assert torch.isclose(result, torch.tensor([[1/15]]), atol=1e-3)
print("✓ 3D IoU test passed!")


3D IoU: 0.0667 (expected 0.0667)
✓ 3D IoU test passed!


## Problem 1D: Vectorized Nearest Neighbor Matching (30 min)

**Two implementations:** Naive (easy to understand) vs Memory-efficient (interview gold).

The key insight: `||a - b||² = ||a||² + ||b||² - 2⟨a, b⟩`

This avoids materializing the `(M, N, D)` difference tensor.


In [79]:
# ============ REFERENCE SOLUTION ============

def nearest_neighbor_naive(points_a: torch.Tensor, points_b: torch.Tensor) -> torch.Tensor:
    """
    Naive version — materializes (M, N, D) tensor.
    Fine when M, N, D are small. OOMs when they're large.
    
    Memory: O(M * N * D)
    """
    # (M, 1, D) - (1, N, D) → (M, N, D) ← this is the memory bottleneck
    diff = points_a.unsqueeze(1) - points_b.unsqueeze(0)
    dist_sq = (diff ** 2).sum(dim=-1)  # (M, N)
    return dist_sq.argmin(dim=-1)      # (M,)


def nearest_neighbor_efficient(points_a: torch.Tensor, points_b: torch.Tensor) -> torch.Tensor:
    """
    Memory-efficient version using the expansion trick.
    
    ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a·b
    
    Memory: O(M * N) — never creates the (M, N, D) tensor
    
    Args:
        points_a: (M, D)
        points_b: (N, D)
    Returns:
        (M,) indices into points_b
    """
    # ||a||^2: (M, 1) — keepdim for broadcasting
    a_sq = (points_a ** 2).sum(dim=-1, keepdim=True)  # (M, 1)
    
    # ||b||^2: (1, N) — transpose for broadcasting
    b_sq = (points_b ** 2).sum(dim=-1).unsqueeze(0)   # (1, N)
    
    # a · b: (M, D) @ (D, N) → (M, N)
    ab = points_a @ points_b.T                         # (M, N)
    
    # ||a - b||^2 = ||a||^2 + ||b||^2 - 2ab
    dist_sq = a_sq + b_sq - 2 * ab                     # (M, N)
    
    return dist_sq.argmin(dim=-1)                       # (M,)


# ============ TEST & COMPARE ============
a = torch.tensor([[0., 0.], [3., 3.], [10., 0.]])
b = torch.tensor([[1., 0.], [3., 4.], [10., 10.]])

result_naive = nearest_neighbor_naive(a, b)
result_efficient = nearest_neighbor_efficient(a, b)
print(f"Naive:     {result_naive.tolist()}")
print(f"Efficient: {result_efficient.tolist()}")
assert result_naive.tolist() == result_efficient.tolist() == [0, 1, 1]

# Memory comparison
M, N, D = 1000, 1000, 256
print(f"\nMemory comparison for M={M}, N={N}, D={D}:")
naive_mem = M * N * D * 4  # float32 = 4 bytes
efficient_mem = M * N * 4
print(f"  Naive (M,N,D) tensor:    {naive_mem / 1e9:.2f} GB")
print(f"  Efficient (M,N) tensor:  {efficient_mem / 1e6:.1f} MB")
print(f"  Reduction factor:        {naive_mem / efficient_mem:.0f}x")

print("\n✓ Both methods match, but efficient uses ~256x less memory!")


Naive:     [0, 1, 1]
Efficient: [0, 1, 1]

Memory comparison for M=1000, N=1000, D=256:
  Naive (M,N,D) tensor:    1.02 GB
  Efficient (M,N) tensor:  4.0 MB
  Reduction factor:        256x

✓ Both methods match, but efficient uses ~256x less memory!


---
# BLOCK 2: Architecture & Sequence Components (~2 hours)


## Problem 2A: Multi-Head Self-Attention from Scratch (40 min)

This is the single most important implementation to nail. Every interviewer loves this one.

**Shape journey:**
```
Input:  (B, S, D)
        ↓ linear projections
Q,K,V:  (B, S, D)
        ↓ reshape + transpose
        (B, H, S, D//H)
        ↓ Q @ K^T / sqrt(d_k)
scores: (B, H, S, S)
        ↓ mask + softmax
attn:   (B, H, S, S)
        ↓ attn @ V
out:    (B, H, S, D//H)
        ↓ reshape
        (B, S, D)
        ↓ output projection
Output: (B, S, D)
```


In [ ]:
# ============ REFERENCE SOLUTION ============
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Q, K, V projections (could also use one big linear + split)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: optional (seq_len, seq_len) bool tensor.
                  True = BLOCK that position (filled with -inf before softmax)
        Returns:
            (batch, seq_len, d_model)
        """
        B, S, D = x.shape
        
        # Step 1: Project to Q, K, V — each (B, S, D)
        Q = self.W_q(x)  # (B, S, D)
        K = self.W_k(x)  # (B, S, D)
        V = self.W_v(x)  # (B, S, D)
        
        # Step 2: Reshape to (B, H, S, d_k) for multi-head parallel computation
        # (B, S, D) → (B, S, H, d_k) → (B, H, S, d_k)
        Q = Q.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, d_k)
        K = K.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, d_k)
        V = V.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, d_k)
        
        # Step 3: Scaled dot-product attention
        # scores = Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, S, S)
        
        # Step 4: Apply mask (if provided)
        if mask is not None:
            # mask shape: (S, S) → broadcast to (B, H, S, S)
            scores = scores.masked_fill(mask, float('-inf'))
        
        # Step 5: Softmax over key dimension
        attn_weights = F.softmax(scores, dim=-1)  # (B, H, S, S)
        
        # Step 6: Weighted sum of values
        out = torch.matmul(attn_weights, V)  # (B, H, S, d_k)
        
        # Step 7: Reshape back to (B, S, D)
        # (B, H, S, d_k) → (B, S, H, d_k) → (B, S, D)
        out = out.transpose(1, 2).contiguous().view(B, S, D)  # (B, S, D)
        
        # Step 8: Output projection
        out = self.W_o(out)  # (B, S, D)
        
        return out

# ============ TESTS ============
mhsa = MultiHeadSelfAttention(d_model=64, num_heads=8)

# Test 1: Basic forward pass
x = torch.randn(2, 10, 64)  # batch=2, seq=10, d_model=64
out = mhsa(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
assert out.shape == (2, 10, 64)
print("✓ Shape correct")

# Test 2: With causal mask
causal_mask = torch.triu(torch.ones(10, 10, dtype=torch.bool), diagonal=1)
out_causal = mhsa(x, mask=causal_mask)
assert out_causal.shape == (2, 10, 64)
print("✓ Causal masked attention works")

# Test 3: Single-head equivalent to standard attention
mhsa_1head = MultiHeadSelfAttention(d_model=16, num_heads=1)
x_small = torch.randn(1, 5, 16)
out_small = mhsa_1head(x_small)
assert out_small.shape == (1, 5, 16)
print("✓ Single-head case works")

print("\n✓ All multi-head attention tests passed!")
print(f"\nComplexity: O(S² · D) time, O(S² · H + S · D) space")
print(f"  S² comes from the attention score matrix")
print(f"  This is why attention is expensive for long sequences")


## Problem 2B: Rotary Positional Embeddings (RoPE) (30 min)

**The math:**
For dimension pair `(2i, 2i+1)`, apply 2D rotation by angle `θ`:
```
x_new[2i]   = x[2i] · cos(θᵢ) − x[2i+1] · sin(θᵢ)
x_new[2i+1] = x[2i] · sin(θᵢ) + x[2i+1] · cos(θᵢ)

where θᵢ = position / 10000^(2i/d)
```

**Key insight for vectorization:** Reshape `x` from `(..., d)` to `(..., d//2, 2)`,
then use cos/sin on the pair dimension.


In [ ]:
# ============ REFERENCE SOLUTION ============
def compute_rope_freqs(d_k: int, max_seq_len: int, base: float = 10000.0) -> torch.Tensor:
    """
    Precompute frequency matrix for RoPE.
    
    Returns:
        freqs: (max_seq_len, d_k // 2) — the θ values
    """
    # Frequency for each dimension pair: 1 / base^(2i/d)
    dim_pairs = torch.arange(0, d_k, 2).float()  # [0, 2, 4, ..., d-2] → (d_k//2,)
    freqs = 1.0 / (base ** (dim_pairs / d_k))     # (d_k//2,)
    
    # Multiply by position indices
    positions = torch.arange(max_seq_len).float()  # (seq_len,)
    
    # Outer product: (seq_len, 1) * (1, d_k//2) → (seq_len, d_k//2)
    angles = positions.unsqueeze(1) * freqs.unsqueeze(0)  # (seq_len, d_k//2)
    
    return angles


def apply_rope(x: torch.Tensor, positions: torch.Tensor = None) -> torch.Tensor:
    """
    Apply rotary positional embeddings.
    
    Args:
        x: (batch, num_heads, seq_len, d_k)
        positions: optional (seq_len,) position indices. 
                   Defaults to 0, 1, 2, ..., seq_len-1
    Returns:
        x with rotary embeddings applied. Same shape.
    """
    B, H, S, D = x.shape
    assert D % 2 == 0, "d_k must be even for RoPE"
    
    # Compute angles
    angles = compute_rope_freqs(D, S)  # (S, D//2)
    
    cos_vals = torch.cos(angles)  # (S, D//2)
    sin_vals = torch.sin(angles)  # (S, D//2)
    
    # Reshape for broadcasting: (1, 1, S, D//2)
    cos_vals = cos_vals.unsqueeze(0).unsqueeze(0)  # (1, 1, S, D//2)
    sin_vals = sin_vals.unsqueeze(0).unsqueeze(0)  # (1, 1, S, D//2)
    
    # Split x into even and odd dimensions
    x_even = x[..., 0::2]  # (B, H, S, D//2) — dimensions 0, 2, 4, ...
    x_odd  = x[..., 1::2]  # (B, H, S, D//2) — dimensions 1, 3, 5, ...
    
    # Apply rotation
    x_even_rot = x_even * cos_vals - x_odd * sin_vals   # (B, H, S, D//2)
    x_odd_rot  = x_even * sin_vals + x_odd * cos_vals   # (B, H, S, D//2)
    
    # Interleave back: stack along last dim then reshape
    # (B, H, S, D//2, 2) → (B, H, S, D)
    x_rot = torch.stack([x_even_rot, x_odd_rot], dim=-1)  # (B, H, S, D//2, 2)
    x_rot = x_rot.view(B, H, S, D)                        # (B, H, S, D)
    
    return x_rot

# ============ TESTS ============
B, H, S, D = 2, 4, 8, 16
Q = torch.randn(B, H, S, D)
K = torch.randn(B, H, S, D)

Q_rot = apply_rope(Q)
K_rot = apply_rope(K)

print(f"Q shape: {Q.shape} → Q_rot shape: {Q_rot.shape}")
assert Q_rot.shape == Q.shape

# Key property of RoPE: dot product Q_rot·K_rot only depends on RELATIVE position
# This is hard to test directly, but we can verify shapes and that rotation preserves norm
q_norm_before = Q[0, 0, 0].norm()
q_norm_after = Q_rot[0, 0, 0].norm()
print(f"\nNorm preservation: before={q_norm_before:.4f}, after={q_norm_after:.4f}")
print(f"Norms approximately equal: {torch.isclose(q_norm_before, q_norm_after, atol=1e-4)}")

print("\n✓ RoPE implementation works!")
print("\nKey insight: RoPE encodes RELATIVE position because")
print("  dot(rotate(q, pos_i), rotate(k, pos_j)) = f(q, k, i-j)")


## Problem 2C: Layer Normalization from Scratch (15 min)

Quick implementation. The formula is simple, but getting the shapes right matters.


In [ ]:
# ============ REFERENCE SOLUTION ============
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))   # scale
        self.beta = nn.Parameter(torch.zeros(normalized_shape))   # shift
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (..., normalized_shape)
        Returns:
            normalized x, same shape
        """
        # Compute stats over last dimension
        mean = x.mean(dim=-1, keepdim=True)      # (..., 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # (..., 1)
        
        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)  # (..., normalized_shape)
        
        # Scale and shift
        return self.gamma * x_norm + self.beta  # (..., normalized_shape)

# ============ TEST: Compare with PyTorch ============
D = 64
our_ln = LayerNorm(D)
torch_ln = nn.LayerNorm(D)

# Copy weights so they match
torch_ln.weight = nn.Parameter(our_ln.gamma.data.clone())
torch_ln.bias = nn.Parameter(our_ln.beta.data.clone())

x = torch.randn(2, 10, D)
out_ours = our_ln(x)
out_torch = torch_ln(x)

print(f"Max diff: {(out_ours - out_torch).abs().max().item():.8f}")
assert torch.allclose(out_ours, out_torch, atol=1e-5)
print("✓ Matches nn.LayerNorm!")


---
# BLOCK 3: High-Efficiency ML Operations (~2 hours)


## Problem 3A: Batched Pairwise Distance — Naive vs Efficient (25 min)

**This is THE pattern for demonstrating you understand memory complexity.**

Interviewers love asking: "what happens at scale?" and this is where you show it.


In [ ]:
# ============ REFERENCE SOLUTION ============

def pairwise_dist_naive(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Naive: materializes (B, M, N, D) difference tensor.
    
    Memory: O(B * M * N * D) ← DANGEROUS at scale
    """
    # x: (B, M, D) → (B, M, 1, D)
    # y: (B, N, D) → (B, 1, N, D)
    diff = x.unsqueeze(2) - y.unsqueeze(1)  # (B, M, N, D) ← memory hog
    return torch.sqrt((diff ** 2).sum(dim=-1))  # (B, M, N)


def pairwise_dist_efficient(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Efficient: uses ||x-y||^2 = ||x||^2 + ||y||^2 - 2*x·y^T
    
    Memory: O(B * M * N) ← D factor eliminated
    """
    # ||x||^2: (B, M, D) → sum → (B, M, 1)
    x_sq = (x ** 2).sum(dim=-1, keepdim=True)  # (B, M, 1)
    
    # ||y||^2: (B, N, D) → sum → (B, 1, N)
    y_sq = (y ** 2).sum(dim=-1).unsqueeze(1)   # (B, 1, N)
    
    # x · y^T: (B, M, D) @ (B, D, N) → (B, M, N)
    xy = torch.bmm(x, y.transpose(1, 2))       # (B, M, N)
    
    # ||x-y||^2 = ||x||^2 + ||y||^2 - 2*x·y^T
    dist_sq = x_sq + y_sq - 2 * xy             # (B, M, N) via broadcasting
    
    # Clamp to avoid negative values from floating point errors
    dist_sq = dist_sq.clamp(min=0)
    
    return torch.sqrt(dist_sq)                  # (B, M, N)


# ============ TEST ============
B, M, N, D = 4, 50, 60, 32
x = torch.randn(B, M, D)
y = torch.randn(B, N, D)

dist_naive = pairwise_dist_naive(x, y)
dist_efficient = pairwise_dist_efficient(x, y)

print(f"Naive shape:     {dist_naive.shape}")
print(f"Efficient shape: {dist_efficient.shape}")
print(f"Max difference:  {(dist_naive - dist_efficient).abs().max().item():.8f}")
assert torch.allclose(dist_naive, dist_efficient, atol=1e-4)
print("✓ Results match!")

# Memory analysis
print(f"\n--- Memory at scale (B=32, M=1000, N=1000, D=256) ---")
B_big, M_big, N_big, D_big = 32, 1000, 1000, 256
naive_bytes = B_big * M_big * N_big * D_big * 4
efficient_bytes = B_big * M_big * N_big * 4
print(f"Naive (B,M,N,D) tensor:   {naive_bytes / 1e9:.1f} GB ← OOM on most GPUs!")
print(f"Efficient (B,M,N) tensor: {efficient_bytes / 1e6:.0f} MB ← fits easily")
print(f"Memory reduction: {naive_bytes / efficient_bytes}x")


## Problem 3B: Top-K Sparse Attention (30 min)

Standard attention has O(S²) memory. For very long sequences, we can attend to only
the top-k most relevant keys per query.


In [ ]:
# ============ REFERENCE SOLUTION ============
def topk_attention(
    Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, k: int
) -> torch.Tensor:
    """
    Sparse attention: each query only attends to top-k keys.
    
    Args:
        Q: (B, H, S, d_k)
        K: (B, H, S, d_k)
        V: (B, H, S, d_k)
        k: number of keys to attend to per query
    Returns:
        (B, H, S, d_k)
    
    Complexity: O(S² · d_k) for score computation (same as full attention)
                but softmax is over k items instead of S → saves on long sequences
    """
    B, H, S, d_k = Q.shape
    
    # Step 1: Full attention scores (we still compute all of them)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, S, S)
    
    # Step 2: Find top-k scores per query
    topk_values, topk_indices = torch.topk(scores, k=k, dim=-1)  # both (B, H, S, k)
    
    # Step 3: Create sparse mask — fill non-top-k positions with -inf
    sparse_scores = torch.full_like(scores, float('-inf'))  # (B, H, S, S)
    sparse_scores.scatter_(-1, topk_indices, topk_values)   # place top-k values back
    
    # Step 4: Softmax (only top-k positions get non-zero weight)
    attn_weights = F.softmax(sparse_scores, dim=-1)  # (B, H, S, S)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)  # (B, H, S, d_k)
    
    return output

# ============ TEST ============
B, H, S, d_k = 2, 4, 20, 16
Q = torch.randn(B, H, S, d_k)
K = torch.randn(B, H, S, d_k)
V = torch.randn(B, H, S, d_k)

out_topk = topk_attention(Q, K, V, k=5)
print(f"Output shape: {out_topk.shape}")
assert out_topk.shape == (B, H, S, d_k)

# When k = S, should equal full attention
out_full_sparse = topk_attention(Q, K, V, k=S)
scores_full = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
attn_full = F.softmax(scores_full, dim=-1)
out_full = torch.matmul(attn_full, V)
print(f"When k=S, max diff from full attention: {(out_full_sparse - out_full).abs().max():.8f}")
assert torch.allclose(out_full_sparse, out_full, atol=1e-5)

print("\n✓ Top-K attention works!")
print(f"\nNote: This doesn't save on the QK^T computation (still O(S²d)).")
print(f"Real sparse attention methods (e.g., reformer, flash attention) avoid this.")
print(f"But this pattern tests your scatter/gather skills.")


## Problem 3C: Efficient Masked Group Aggregation (20 min)

**Common pattern:** You have N items assigned to G groups. Compute per-group statistics without loops.


In [ ]:
# ============ REFERENCE SOLUTION ============
def group_mean(
    embeddings: torch.Tensor, group_ids: torch.Tensor, num_groups: int
) -> torch.Tensor:
    """
    Compute mean embedding per group without loops.
    
    Args:
        embeddings: (N, D) token embeddings
        group_ids: (N,) integer group assignment [0, num_groups)
        num_groups: total groups
    Returns:
        (num_groups, D) mean embedding per group
    """
    D = embeddings.shape[1]
    
    # Expand group_ids for scatter: (N,) → (N, D)
    ids_expanded = group_ids.unsqueeze(1).expand(-1, D)  # (N, D)
    
    # Sum per group
    group_sums = torch.zeros(num_groups, D, dtype=embeddings.dtype)
    group_sums.scatter_add_(0, ids_expanded, embeddings)  # (num_groups, D)
    
    # Count per group
    counts = torch.bincount(group_ids, minlength=num_groups).float()  # (num_groups,)
    counts = counts.clamp(min=1)  # avoid division by zero for empty groups
    
    return group_sums / counts.unsqueeze(1)  # (num_groups, D)

# ============ TEST ============
emb = torch.tensor([[1., 0.], [2., 0.], [0., 3.], [0., 1.], [5., 5.]])
ids = torch.tensor([0, 0, 1, 1, 2])

result = group_mean(emb, ids, num_groups=3)
expected = torch.tensor([[1.5, 0.], [0., 2.], [5., 5.]])
print(f"Group means:\n{result}")
assert torch.allclose(result, expected)
print("✓ Group mean test passed!")

# Bonus: what about empty groups?
result2 = group_mean(emb[:4], ids[:4], num_groups=3)
print(f"\nWith empty group 2:\n{result2}")
# Group 2 should be zeros (0/1 = 0)
print("✓ Handles empty groups gracefully")


---
# Quick Reference: Common Tensor Operations Cheat Sheet

## Reshaping
```python
x.view(B, S, H, D)          # reshape (must be contiguous)
x.reshape(B, S, H, D)       # reshape (always works, may copy)
x.transpose(1, 2)           # swap dims 1 and 2
x.permute(0, 2, 1, 3)       # arbitrary dim reordering
x.unsqueeze(1)               # add dim at position 1
x.squeeze(1)                 # remove dim at position 1 (if size 1)
x.contiguous()               # make memory contiguous (needed before view after transpose)
x[..., 0::2]                 # slice every other element along last dim
```

## Broadcasting
```python
x.unsqueeze(1) - y.unsqueeze(0)   # (M,1,D) - (1,N,D) → (M,N,D) pairwise
x + y.unsqueeze(-1)               # add with extra trailing dim
```

## Reductions
```python
x.sum(dim=-1)                    # sum over last dim → removes it
x.sum(dim=-1, keepdim=True)      # sum over last dim → keeps as size 1
x.mean(dim=-1)                   # mean
x.max(dim=-1)                    # returns (values, indices) named tuple
x.argmax(dim=-1)                 # just indices
x.norm(dim=-1)                   # L2 norm
```

## Masking
```python
scores.masked_fill_(mask, float('-inf'))  # fill where mask is True
torch.where(condition, x, y)              # element-wise conditional
x[mask]                                    # boolean indexing (returns 1D)
```

## Scatter / Gather
```python
out.scatter_add_(dim, index, src)    # add src values into out at index positions
torch.gather(input, dim, index)      # select values at index positions
torch.topk(x, k, dim=-1)            # top-k values and indices
```

## Einsum
```python
torch.einsum('bik,bkj->bij', A, B)     # batch matmul
torch.einsum('bhqd,bhkd->bhqk', Q, K)  # attention scores
torch.einsum('ij,j->i', A, v)          # matrix-vector product
torch.einsum('ii->', A)                 # trace
```


---
## Done!

**Recommended order:**
1. Level 0: Broadcasting drills (30 min) — if you need the refresher
2. Level 1: Core patterns (45 min) — build the muscle memory
3. Block 1: Problems 1A → 1B → 1D (90 min) — highest priority for embodied AI
4. Block 2: Problem 2A (40 min) — the must-know implementation
5. Block 3: Problem 3A (25 min) — the memory efficiency showpiece
6. Remaining problems as time allows

**If pressed for time, the minimum set is:** 1A, 2A, 3A — those three cover 80% of what interviewers test.
